'predict_answer.ipynb'를 실행하여 예측 테스트를 진행하세요.

-> **README.md에 있는 링크에서 answer_1, answer_2 이미지를 다운받아 CNN 폴더에 넣은 후 코드를 실행하세요**


-> **kaggle 오류가 발생할 경우, kaggle > setting에서 kaggle.json을 다운받아 CNN 폴더에 넣은 뒤 환경변수 설정 코드를 주석 해제 후 실행하세요.**

In [ ]:
# import os
# import json

# # 1️⃣ kaggle.json 파일 경로 지정
# kaggle_json_path = os.path.expanduser("kaggle.json")

# # 2️⃣ kaggle.json 파일이 존재하는지 확인
# if not os.path.exists(kaggle_json_path):
#     raise FileNotFoundError(
#         f"kaggle.json 파일을 찾을 수 없습니다: {kaggle_json_path}\n"
#     )

# # 3️⃣ kaggle.json 내용 읽기
# with open(kaggle_json_path, "r") as f:
#     kaggle_config = json.load(f)

# # 4️⃣ os 환경 변수 설정
# os.environ["KAGGLE_USERNAME"] = kaggle_config.get("username")
# os.environ["KAGGLE_KEY"] = kaggle_config.get("key")

# print("✅ Kaggle API 환경변수가 설정되었습니다.")
# print(f"USERNAME: {os.environ['KAGGLE_USERNAME']}")

✅ Kaggle API 환경변수가 설정되었습니다.
USERNAME: harukaggler


In [2]:
import torch
import torch.nn as nn
from PIL import Image
import numpy as np
from pathlib import Path
import logging
import os

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# ========== Model Architectures ==========

class SEBlock(nn.Module):
    """Squeeze-and-Excitation block"""
    def __init__(self, in_channels, se_channels):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(in_channels, se_channels, bias=False),
            nn.SiLU(inplace=True),
            nn.Linear(se_channels, in_channels, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _, _ = x.size()
        y = self.avg_pool(x).view(b, c)
        y = self.fc(y).view(b, c, 1, 1)
        return x * y


class ResidualBlock(nn.Module):
    """Residual connection wrapper"""
    def __init__(self, module):
        super().__init__()
        self.module = module

    def forward(self, x):
        return x + self.module(x)


class BasicBlock(nn.Module):
    """Basic block for ResNet"""
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()

        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, stride, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)

        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, 1, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)

        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 1, stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )

    def forward(self, x):
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        out = self.relu(out)
        return out


class InvertedResidual(nn.Module):
    """Inverted Residual block for MobileNetV3"""
    def __init__(self, in_channels, out_channels, kernel, stride, expand_ratio, se_ratio=None):
        super().__init__()

        hidden_dim = int(in_channels * expand_ratio)
        self.use_res_connect = stride == 1 and in_channels == out_channels

        layers = []

        if expand_ratio != 1:
            layers.extend([
                nn.Conv2d(in_channels, hidden_dim, 1, bias=False),
                nn.BatchNorm2d(hidden_dim),
                nn.Hardswish(inplace=True)
            ])

        layers.extend([
            nn.Conv2d(hidden_dim, hidden_dim, kernel, stride,
                     kernel//2, groups=hidden_dim, bias=False),
            nn.BatchNorm2d(hidden_dim),
            nn.Hardswish(inplace=True)
        ])

        if se_ratio is not None:
            se_channels = int(in_channels * se_ratio)
            layers.append(SEBlock(hidden_dim, se_channels))

        layers.extend([
            nn.Conv2d(hidden_dim, out_channels, 1, bias=False),
            nn.BatchNorm2d(out_channels)
        ])

        self.conv = nn.Sequential(*layers)

    def forward(self, x):
        if self.use_res_connect:
            return x + self.conv(x)
        else:
            return self.conv(x)


class EfficientNetB0_MNIST(nn.Module):
    """EfficientNet-B0 for MNIST"""
    def __init__(self, num_classes=10):
        super().__init__()

        self.conv_stem = nn.Conv2d(1, 32, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(32)
        self.act1 = nn.SiLU(inplace=True)

        self.blocks = nn.Sequential(
            self._make_mbconv(32, 16, kernel=3, stride=1, expand_ratio=1),
            self._make_mbconv(16, 24, kernel=3, stride=2, expand_ratio=6),
            self._make_mbconv(24, 24, kernel=3, stride=1, expand_ratio=6),
            self._make_mbconv(24, 40, kernel=5, stride=2, expand_ratio=6),
            self._make_mbconv(40, 40, kernel=5, stride=1, expand_ratio=6),
            self._make_mbconv(40, 80, kernel=3, stride=1, expand_ratio=6),
            self._make_mbconv(80, 80, kernel=3, stride=1, expand_ratio=6),
            self._make_mbconv(80, 112, kernel=5, stride=1, expand_ratio=6),
            self._make_mbconv(112, 112, kernel=5, stride=1, expand_ratio=6),
            self._make_mbconv(112, 192, kernel=5, stride=1, expand_ratio=6),
        )

        self.conv_head = nn.Conv2d(192, 1280, kernel_size=1, bias=False)
        self.bn2 = nn.BatchNorm2d(1280)
        self.act2 = nn.SiLU(inplace=True)

        self.global_pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Dropout(0.2),
            nn.Linear(1280, num_classes)
        )

    def _make_mbconv(self, in_channels, out_channels, kernel, stride, expand_ratio):
        layers = []
        hidden_dim = in_channels * expand_ratio

        if expand_ratio != 1:
            layers.extend([
                nn.Conv2d(in_channels, hidden_dim, 1, bias=False),
                nn.BatchNorm2d(hidden_dim),
                nn.SiLU(inplace=True)
            ])

        layers.extend([
            nn.Conv2d(hidden_dim, hidden_dim, kernel, stride,
                     padding=kernel//2, groups=hidden_dim, bias=False),
            nn.BatchNorm2d(hidden_dim),
            nn.SiLU(inplace=True)
        ])

        se_channels = max(1, in_channels // 4)
        layers.append(SEBlock(hidden_dim, se_channels))

        layers.extend([
            nn.Conv2d(hidden_dim, out_channels, 1, bias=False),
            nn.BatchNorm2d(out_channels)
        ])

        if stride == 1 and in_channels == out_channels:
            return ResidualBlock(nn.Sequential(*layers))

        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.conv_stem(x)
        x = self.bn1(x)
        x = self.act1(x)
        x = self.blocks(x)
        x = self.conv_head(x)
        x = self.bn2(x)
        x = self.act2(x)
        x = self.global_pool(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x


class ResNet18_MNIST(nn.Module):
    """ResNet-18 for MNIST"""
    def __init__(self, num_classes=10):
        super().__init__()

        self.conv1 = nn.Conv2d(1, 64, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU(inplace=True)

        self.layer1 = self._make_layer(64, 64, 2, stride=1)
        self.layer2 = self._make_layer(64, 128, 2, stride=2)
        self.layer3 = self._make_layer(128, 256, 2, stride=2)
        self.layer4 = self._make_layer(256, 512, 2, stride=2)

        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512, num_classes)

    def _make_layer(self, in_channels, out_channels, blocks, stride=1):
        layers = []
        layers.append(BasicBlock(in_channels, out_channels, stride))
        for _ in range(1, blocks):
            layers.append(BasicBlock(out_channels, out_channels, stride=1))
        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)
        return x


class MobileNetV3_MNIST(nn.Module):
    """MobileNetV3-Small for MNIST"""
    def __init__(self, num_classes=10):
        super().__init__()

        self.conv_stem = nn.Conv2d(1, 16, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(16)
        self.act1 = nn.Hardswish(inplace=True)

        self.blocks = nn.Sequential(
            InvertedResidual(16, 16, kernel=3, stride=1, expand_ratio=1, se_ratio=0.25),
            InvertedResidual(16, 24, kernel=3, stride=2, expand_ratio=4.5, se_ratio=None),
            InvertedResidual(24, 24, kernel=3, stride=1, expand_ratio=3.67, se_ratio=None),
            InvertedResidual(24, 40, kernel=5, stride=2, expand_ratio=4, se_ratio=0.25),
            InvertedResidual(40, 40, kernel=5, stride=1, expand_ratio=6, se_ratio=0.25),
            InvertedResidual(40, 40, kernel=5, stride=1, expand_ratio=6, se_ratio=0.25),
            InvertedResidual(40, 48, kernel=5, stride=1, expand_ratio=3, se_ratio=0.25),
            InvertedResidual(48, 48, kernel=5, stride=1, expand_ratio=3, se_ratio=0.25),
            InvertedResidual(48, 96, kernel=5, stride=1, expand_ratio=6, se_ratio=0.25),
            InvertedResidual(96, 96, kernel=5, stride=1, expand_ratio=6, se_ratio=0.25),
        )

        self.conv_head = nn.Conv2d(96, 576, kernel_size=1, bias=False)
        self.bn2 = nn.BatchNorm2d(576)
        self.act2 = nn.Hardswish(inplace=True)

        self.global_pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Linear(576, 1024),
            nn.Hardswish(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(1024, num_classes)
        )

    def forward(self, x):
        x = self.conv_stem(x)
        x = self.bn1(x)
        x = self.act1(x)
        x = self.blocks(x)
        x = self.conv_head(x)
        x = self.bn2(x)
        x = self.act2(x)
        x = self.global_pool(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x


# ========== Kaggle Download Function ==========

def download_models_from_kaggle(dataset_id='minyujin03/mnist-finetuned-models', 
                                 download_path='./kaggle_models'):
    """Kaggle에서 fine-tuned 모델 다운로드"""
    
    download_path = Path(download_path)
    
    # 이미 다운로드되어 있는지 확인
    if download_path.exists() and len(list(download_path.glob('*.pth'))) == 6:
        print(f"✅ 모델이 이미 다운로드되어 있습니다: {download_path}")
        return download_path
    
    print(f"Kaggle에서 모델 다운로드 중")
    print(f"Dataset: {dataset_id}")
    
    try:
        from kaggle.api.kaggle_api_extended import KaggleApi
        
        api = KaggleApi()
        api.authenticate()
        
        # 다운로드 폴더 생성
        download_path.mkdir(parents=True, exist_ok=True)
        
        # 데이터셋 다운로드
        api.dataset_download_files(
            dataset_id,
            path=str(download_path),
            unzip=True
        )
        
        # 다운로드된 파일 확인
        downloaded_files = list(download_path.glob('*.pth'))
        print(f"✅ 다운로드 완료! {len(downloaded_files)}개 파일")
        for file in downloaded_files:
            size_mb = file.stat().st_size / (1024 * 1024)
            print(f"   - {file.name} ({size_mb:.2f} MB)")
        
        return download_path
        
    except Exception as e:
        print(f"\n Kaggle 다운로드 실패: {e}")
        print("\n 해결 방법:")
        print("   1. Kaggle API가 설치되어 있는지 확인: pip install kaggle")
        print("   2. ~/.kaggle/kaggle.json 파일이 있는지 확인")
        print("   3. Dataset이 Public인지 확인")
        raise


# ========== Model Loading ==========

def load_answer_model(model_path, model_type, device='cpu'):
    """학습된 Answer 모델 로드"""

    try:
        checkpoint = torch.load(model_path, map_location=device, weights_only=False)

        # CV 결과 파일 형식 (model_type 포함)
        if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint and 'model_type' in checkpoint:
            loaded_model_type = checkpoint['model_type']
            model_state_dict = checkpoint['model_state_dict']
            best_cv_acc = checkpoint.get('best_cv_acc', 'N/A')

            if loaded_model_type == 'resnet':
                model = ResNet18_MNIST(num_classes=5) 
            elif loaded_model_type == 'efficientnet':
                model = EfficientNetB0_MNIST(num_classes=5)
            elif loaded_model_type == 'mobilenet':
                model = MobileNetV3_MNIST(num_classes=5)
            else:
                raise ValueError(f"Unsupported model type: {loaded_model_type}")

            model.load_state_dict(model_state_dict, strict=False)
            print(f"   Model loaded from CV checkpoint")
            if best_cv_acc != 'N/A':
                print(f"   Best CV Validation Acc: {best_cv_acc:.2f}%")

        # 구 형식 (model_state_dict만)
        elif isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
            if model_type == 'resnet':
                model = ResNet18_MNIST(num_classes=5)
            elif model_type == 'efficientnet':
                model = EfficientNetB0_MNIST(num_classes=5)
            elif model_type == 'mobilenet':
                model = MobileNetV3_MNIST(num_classes=5)
            else:
                raise ValueError(f"Unsupported model type: {model_type}")

            model.load_state_dict(checkpoint['model_state_dict'], strict=False)
            print(f"   Model loaded from old checkpoint")
            if 'val_acc' in checkpoint:
                print(f"   Validation Acc: {checkpoint['val_acc']:.2f}%")

        # Raw state_dict
        else:
            if model_type == 'resnet':
                model = ResNet18_MNIST(num_classes=5)
            elif model_type == 'efficientnet':
                model = EfficientNetB0_MNIST(num_classes=5)
            elif model_type == 'mobilenet':
                model = MobileNetV3_MNIST(num_classes=5)
            else:
                raise ValueError(f"Unsupported model type: {model_type}")

            model.load_state_dict(checkpoint, strict=False)
            print(f"   Raw state_dict loaded")

    except RuntimeError as e:
        if "size mismatch" in str(e):
            logger.error(f"❌ Size mismatch: {model_path}")
            print("💡 모델 아키텍처가 저장된 모델과 일치하지 않습니다")
        else:
            logger.error(f"❌ Failed to load: {model_path}. Error: {e}")
        raise e 

    model = model.to(device)
    model.eval()

    print(f"   {model_type.upper()} 모델 로드 완료")

    return model


def preprocess_image(image_path):
    """이미지 전처리"""
    image = Image.open(image_path).convert('L')
    image = image.resize((28, 28))
    image = torch.FloatTensor(np.array(image)).unsqueeze(0).unsqueeze(0) / 255.0
    image = (image - 0.1307) / 0.3081
    return image


def predict_single_image(model, image_path, device='cpu'):
    """단일 이미지 예측"""
    image = preprocess_image(image_path).to(device)

    with torch.no_grad():
        output = model(image)
        _, predicted = output.max(1)
        prediction = predicted.item() + 1  

    return prediction


def evaluate_dataset(model, data_dir, model_type, answer_type, device):
    """데이터셋 전체 평가"""

    total_correct = 0
    total_images = 0

    print(f"\n--- {answer_type} / {model_type.upper()} 테스트 시작 ---")

    # 각 클래스별 평가
    for class_idx in range(1, 6):
        class_dir = data_dir / str(class_idx)
        if not class_dir.exists():
            print(f"  ⚠️  클래스 {class_idx} 폴더 없음")
            continue

        image_paths = list(class_dir.glob("*.jpg")) + list(class_dir.glob("*.png"))
        correct = 0

        for img_path in image_paths:
            prediction = predict_single_image(model, img_path, device)
            if prediction == class_idx:
                correct += 1
            else:
                print(f"    ❌ {img_path.name}: 예측={prediction}, 실제={class_idx}")

        accuracy = 100. * correct / len(image_paths) if len(image_paths) > 0 else 0
        print(f"  클래스 {class_idx}: {correct}/{len(image_paths)} 정답 ({accuracy:.2f}%)")

        total_correct += correct
        total_images += len(image_paths)

    overall_accuracy = 100. * total_correct / total_images if total_images > 0 else 0
    print(f"--- 전체 정확도: {overall_accuracy:.2f}% ({total_correct}/{total_images}) ---")
    
    return overall_accuracy, total_correct, total_images


# ========== Main Execution ==========

def main():
    """메인 실행"""

    print("="*70)
    print("Fine-tuned 모델 예측 테스트 (Kaggle 모델 사용)")
    print("="*70)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"\n디바이스: {device}")

    # 1. Kaggle에서 모델 다운로드
    print(f"\n{'='*70}")
    print(f"Step 1: Kaggle에서 모델 다운로드")
    print(f"{'='*70}")
    
    KAGGLE_DATASET_ID = 'minyujin03/mnist-finetuned-models'
    model_root = download_models_from_kaggle(
        dataset_id=KAGGLE_DATASET_ID,
        download_path='./kaggle_models'
    )

    # 2. 로컬 데이터 경로 설정
    base_dir = Path(".")
    data_root = base_dir  # answer_1, answer_2 폴더가 있는 위치

    # 3. 평가할 모델 목록
    models_to_evaluate = [
        {'answer': 'answer_1', 'type': 'resnet'},
        {'answer': 'answer_2', 'type': 'resnet'},
        {'answer': 'answer_1', 'type': 'efficientnet'}, 
        {'answer': 'answer_2', 'type': 'efficientnet'},
        {'answer': 'answer_1', 'type': 'mobilenet'},
        {'answer': 'answer_2', 'type': 'mobilenet'},
    ]

    results = {}

    # 4. 각 모델 평가
    print(f"\n{'='*70}")
    print(f"Step 2: 모델 평가")
    print(f"{'='*70}")

    for item in models_to_evaluate:
        answer_type = item['answer']
        model_type = item['type']
        
        # Kaggle에서 다운로드한 모델 경로
        model_filename = f"{answer_type}_{model_type}.pth"
        model_path = model_root / model_filename

        # 로컬 데이터 경로
        data_dir = data_root / answer_type

        if not model_path.exists():
            print(f"\n❌ 모델 파일 없음: {model_path}")
            continue

        if not data_dir.exists():
            print(f"\n❌ 데이터 폴더 없음: {data_dir}")
            print(f"{answer_type} 폴더가 현재 디렉토리에 있는지 확인하세요")
            continue

        try:
            print(f"\n{'#'*60}")
            print(f"📊 {answer_type} / {model_type.upper()} 평가")
            print(f"{'#'*60}")
            print(f"   모델: {model_path}")
            print(f"   데이터: {data_dir}")
            
            model = load_answer_model(model_path, model_type=model_type, device=device)
            acc, correct, total = evaluate_dataset(model, data_dir, model_type, answer_type, device)
            
            results[f"{answer_type}_{model_type}"] = {
                'acc': acc, 
                'correct': correct, 
                'total': total
            }
            
        except Exception as e:
            print(f"❌ 평가 중 오류: {e}")
            import traceback
            traceback.print_exc()

    # 5. 최종 결과 요약
    print(f"\n{'='*70}")
    print("🏆 최종 성능 요약")
    print(f"{'='*70}")

    for answer_type in ['answer_1', 'answer_2']:
        print(f"\n📊 {answer_type} 성능:")
        evaluated_models = [
            m_type for m_type in ['resnet', 'efficientnet', 'mobilenet'] 
            if f"{answer_type}_{m_type}" in results
        ]
        
        if evaluated_models:
            for model_type in evaluated_models:
                key = f"{answer_type}_{model_type}"
                res = results[key]
                print(f"  {model_type.upper():12s}: {res['acc']:6.2f}% ({res['correct']}/{res['total']})")
        else:
            print("  평가된 모델 없음")

    print(f"\n{'='*70}")
    print("모든 평가를 완료했습니다!")
    print(f"{'='*70}")


if __name__ == "__main__":
    main()

Fine-tuned 모델 예측 테스트 (Kaggle 모델 사용)

디바이스: cuda

Step 1: Kaggle에서 모델 다운로드
Kaggle에서 모델 다운로드 중
Dataset: minyujin03/mnist-finetuned-models
Dataset URL: https://www.kaggle.com/datasets/minyujin03/mnist-finetuned-models
✅ 다운로드 완료! 6개 파일
   - answer_1_resnet.pth (42.69 MB)
   - answer_2_efficientnet.pth (4.15 MB)
   - answer_2_resnet.pth (42.69 MB)
   - answer_1_mobilenet.pth (3.86 MB)
   - answer_1_efficientnet.pth (4.15 MB)
   - answer_2_mobilenet.pth (3.86 MB)

Step 2: 모델 평가

############################################################
📊 answer_1 / RESNET 평가
############################################################
   모델: kaggle_models/answer_1_resnet.pth
   데이터: answer_1
   Model loaded from CV checkpoint
   Best CV Validation Acc: 94.20%
   RESNET 모델 로드 완료

--- answer_1 / RESNET 테스트 시작 ---
  클래스 1: 189/189 정답 (100.00%)
    ❌ 2022 실전 5회 - 3_answer_1_029_conf0.685.jpg: 예측=3, 실제=2
    ❌ 2023 실전 - 33_answer_1_090_conf0.245.jpg: 예측=1, 실제=2
  클래스 2: 206/208 정답 (99.04%)
    ❌ 2023 실전 - 33_a

아래 셀은 실행할 수 없습니다.

In [2]:
"""
import torch
import torch.nn as nn
from PIL import Image
import numpy as np
from pathlib import Path
import logging

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# ========== Model Architectures ==========

class SEBlock(nn.Module):
    """Squeeze-and-Excitation block"""
    def __init__(self, in_channels, se_channels):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(in_channels, se_channels, bias=False),
            nn.SiLU(inplace=True),
            nn.Linear(se_channels, in_channels, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _, _ = x.size()
        y = self.avg_pool(x).view(b, c)
        y = self.fc(y).view(b, c, 1, 1)
        return x * y


class ResidualBlock(nn.Module):
    """Residual connection wrapper"""
    def __init__(self, module):
        super().__init__()
        self.module = module

    def forward(self, x):
        return x + self.module(x)


class BasicBlock(nn.Module):
    """Basic block for ResNet"""
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()

        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, stride, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)

        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, 1, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)

        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 1, stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )

    def forward(self, x):
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        out = self.relu(out)
        return out


class InvertedResidual(nn.Module):
    """Inverted Residual block for MobileNetV3"""
    def __init__(self, in_channels, out_channels, kernel, stride, expand_ratio, se_ratio=None):
        super().__init__()

        hidden_dim = int(in_channels * expand_ratio)
        self.use_res_connect = stride == 1 and in_channels == out_channels

        layers = []

        if expand_ratio != 1:
            layers.extend([
                nn.Conv2d(in_channels, hidden_dim, 1, bias=False),
                nn.BatchNorm2d(hidden_dim),
                nn.Hardswish(inplace=True)
            ])

        layers.extend([
            nn.Conv2d(hidden_dim, hidden_dim, kernel, stride,
                     kernel//2, groups=hidden_dim, bias=False),
            nn.BatchNorm2d(hidden_dim),
            nn.Hardswish(inplace=True)
        ])

        if se_ratio is not None:
            se_channels = int(in_channels * se_ratio)
            layers.append(SEBlock(hidden_dim, se_channels))

        layers.extend([
            nn.Conv2d(hidden_dim, out_channels, 1, bias=False),
            nn.BatchNorm2d(out_channels)
        ])

        self.conv = nn.Sequential(*layers)

    def forward(self, x):
        if self.use_res_connect:
            return x + self.conv(x)
        else:
            return self.conv(x)


class EfficientNetB0_MNIST(nn.Module):
    """EfficientNet-B0 for MNIST"""
    def __init__(self, num_classes=10):
        super().__init__()

        self.conv_stem = nn.Conv2d(1, 32, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(32)
        self.act1 = nn.SiLU(inplace=True)

        self.blocks = nn.Sequential(
            self._make_mbconv(32, 16, kernel=3, stride=1, expand_ratio=1),
            self._make_mbconv(16, 24, kernel=3, stride=2, expand_ratio=6),
            self._make_mbconv(24, 24, kernel=3, stride=1, expand_ratio=6),
            self._make_mbconv(24, 40, kernel=5, stride=2, expand_ratio=6),
            self._make_mbconv(40, 40, kernel=5, stride=1, expand_ratio=6),
            self._make_mbconv(40, 80, kernel=3, stride=1, expand_ratio=6),
            self._make_mbconv(80, 80, kernel=3, stride=1, expand_ratio=6),
            self._make_mbconv(80, 112, kernel=5, stride=1, expand_ratio=6),
            self._make_mbconv(112, 112, kernel=5, stride=1, expand_ratio=6),
            self._make_mbconv(112, 192, kernel=5, stride=1, expand_ratio=6),
        )

        self.conv_head = nn.Conv2d(192, 1280, kernel_size=1, bias=False)
        self.bn2 = nn.BatchNorm2d(1280)
        self.act2 = nn.SiLU(inplace=True)

        self.global_pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Dropout(0.2),
            nn.Linear(1280, num_classes)
        )

    def _make_mbconv(self, in_channels, out_channels, kernel, stride, expand_ratio):
        layers = []
        hidden_dim = in_channels * expand_ratio

        if expand_ratio != 1:
            layers.extend([
                nn.Conv2d(in_channels, hidden_dim, 1, bias=False),
                nn.BatchNorm2d(hidden_dim),
                nn.SiLU(inplace=True)
            ])

        layers.extend([
            nn.Conv2d(hidden_dim, hidden_dim, kernel, stride,
                     padding=kernel//2, groups=hidden_dim, bias=False),
            nn.BatchNorm2d(hidden_dim),
            nn.SiLU(inplace=True)
        ])

        se_channels = max(1, in_channels // 4)
        layers.append(SEBlock(hidden_dim, se_channels))

        layers.extend([
            nn.Conv2d(hidden_dim, out_channels, 1, bias=False),
            nn.BatchNorm2d(out_channels)
        ])

        if stride == 1 and in_channels == out_channels:
            return ResidualBlock(nn.Sequential(*layers))

        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.conv_stem(x)
        x = self.bn1(x)
        x = self.act1(x)
        x = self.blocks(x)
        x = self.conv_head(x)
        x = self.bn2(x)
        x = self.act2(x)
        x = self.global_pool(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x


class ResNet18_MNIST(nn.Module):
    """ResNet-18 for MNIST"""
    def __init__(self, num_classes=10):
        super().__init__()

        self.conv1 = nn.Conv2d(1, 64, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU(inplace=True)

        self.layer1 = self._make_layer(64, 64, 2, stride=1)
        self.layer2 = self._make_layer(64, 128, 2, stride=2)
        self.layer3 = self._make_layer(128, 256, 2, stride=2)
        self.layer4 = self._make_layer(256, 512, 2, stride=2)

        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512, num_classes)

    def _make_layer(self, in_channels, out_channels, blocks, stride=1):
        layers = []
        layers.append(BasicBlock(in_channels, out_channels, stride))
        for _ in range(1, blocks):
            layers.append(BasicBlock(out_channels, out_channels, stride=1))
        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)
        return x


class MobileNetV3_MNIST(nn.Module):
    """MobileNetV3-Small for MNIST"""
    def __init__(self, num_classes=10):
        super().__init__()

        self.conv_stem = nn.Conv2d(1, 16, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(16)
        self.act1 = nn.Hardswish(inplace=True)

        self.blocks = nn.Sequential(
            InvertedResidual(16, 16, kernel=3, stride=1, expand_ratio=1, se_ratio=0.25),
            InvertedResidual(16, 24, kernel=3, stride=2, expand_ratio=4.5, se_ratio=None),
            InvertedResidual(24, 24, kernel=3, stride=1, expand_ratio=3.67, se_ratio=None),
            InvertedResidual(24, 40, kernel=5, stride=2, expand_ratio=4, se_ratio=0.25),
            InvertedResidual(40, 40, kernel=5, stride=1, expand_ratio=6, se_ratio=0.25),
            InvertedResidual(40, 40, kernel=5, stride=1, expand_ratio=6, se_ratio=0.25),
            InvertedResidual(40, 48, kernel=5, stride=1, expand_ratio=3, se_ratio=0.25),
            InvertedResidual(48, 48, kernel=5, stride=1, expand_ratio=3, se_ratio=0.25),
            InvertedResidual(48, 96, kernel=5, stride=1, expand_ratio=6, se_ratio=0.25),
            InvertedResidual(96, 96, kernel=5, stride=1, expand_ratio=6, se_ratio=0.25),
        )

        self.conv_head = nn.Conv2d(96, 576, kernel_size=1, bias=False)
        self.bn2 = nn.BatchNorm2d(576)
        self.act2 = nn.Hardswish(inplace=True)

        self.global_pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Linear(576, 1024),
            nn.Hardswish(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(1024, num_classes)
        )

    def forward(self, x):
        x = self.conv_stem(x)
        x = self.bn1(x)
        x = self.act1(x)
        x = self.blocks(x)
        x = self.conv_head(x)
        x = self.bn2(x)
        x = self.act2(x)
        x = self.global_pool(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x

#=================================================


def load_answer_model(model_path, model_type, device='cpu'):
    """학습된 Answer 모델 로드"""

    try:
        
        checkpoint = torch.load(model_path, map_location=device, weights_only=False)

        
        if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint and 'model_type' in checkpoint:
             loaded_model_type = checkpoint['model_type']
             model_state_dict = checkpoint['model_state_dict']
             best_cv_acc = checkpoint.get('best_cv_acc', 'N/A')

             
             if loaded_model_type == 'resnet':
                 model = ResNet18_MNIST(num_classes=5) 
             elif loaded_model_type == 'efficientnet':
                 model = EfficientNetB0_MNIST(num_classes=5)
             elif loaded_model_type == 'mobilenet':
                 model = MobileNetV3_MNIST(num_classes=5)
             else:
                 raise ValueError(f"Unsupported model type found in checkpoint: {loaded_model_type}")

             model.load_state_dict(model_state_dict, strict=False)
             print(f"Model state_dict loaded from CV result file")
             if best_cv_acc != 'N/A':
                 print(f"   - Best CV Validation Acc: {best_cv_acc:.2f}%")


        elif isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
             if model_type == 'resnet':
                  model = ResNet18_MNIST(num_classes=5)
             elif model_type == 'efficientnet':
                  model = EfficientNetB0_MNIST(num_classes=5)
             elif model_type == 'mobilenet':
                  model = MobileNetV3_MNIST(num_classes=5)
             else:
                  raise ValueError(f"Unsupported model type specified: {model_type}")

             model.load_state_dict(checkpoint['model_state_dict'], strict=False)
             print(f"Model state_dict loaded from old checkpoint format")
             if 'val_acc' in checkpoint:
                  print(f"   - Validation Acc: {checkpoint['val_acc']:.2f}%")

        else:
             if model_type == 'resnet':
                  model = ResNet18_MNIST(num_classes=5)
             elif model_type == 'efficientnet':
                  model = EfficientNetB0_MNIST(num_classes=5)
             elif model_type == 'mobilenet':
                  model = MobileNetV3_MNIST(num_classes=5)
             else:
                  raise ValueError(f"Unsupported model type specified for raw state_dict loading: {model_type}")

             model.load_state_dict(checkpoint, strict=False)
             print(f"Raw model state_dict loaded")


    except RuntimeError as e:
        if "size mismatch" in str(e):
            logger.error(f"❌ Size mismatch error when loading {model_type.upper()} model from {model_path}. Saved model architecture likely differs from the instantiated one. Error: {e}")
            print("\n💡 Hint: The saved model file might not match the expected architecture. Please verify the model saved in this path.")
        else:
             logger.error(f"❌ Failed to load model state dict for {model_type.upper()} from {model_path}. Error: {e}")
        raise e 


    model = model.to(device)
    model.eval()

    print(f"{model_type.upper()} 모델 로드 완료: {model_path}")


    return model


def preprocess_image(image_path):
    """이미지 전처리 함수"""
    image = Image.open(image_path).convert('L')
    image = image.resize((28, 28))
    image = torch.FloatTensor(np.array(image)).unsqueeze(0).unsqueeze(0) / 255.0
    image = (image - 0.1307) / 0.3081
    return image


def predict_single_image(model, image_path, device='cpu'):
    """이미지 클래스 예측"""
    image = preprocess_image(image_path).to(device)

    with torch.no_grad():
        output = model(image)
        _, predicted = output.max(1)
        prediction = predicted.item() + 1  

    return prediction


def evaluate_dataset(model, data_dir, model_type, answer_type, device):
    """특정 Answer 데이터셋 전체 평가"""

    total_correct = 0
    total_images = 0

    print(f"\n--- {answer_type} / {model_type.upper()} 테스트 시작 ---")

    # 각 클래스별로 평가
    for class_idx in range(1, 6):
        class_dir = data_dir / str(class_idx)
        if not class_dir.exists():
            print(f"클래스 {class_idx} 폴더 없음")
            continue

        image_paths = list(class_dir.glob("*.jpg")) + list(class_dir.glob("*.png"))
        correct = 0


        for img_path in image_paths:
            prediction = predict_single_image(model, img_path, device)
            if prediction == class_idx:
                correct += 1
            else:
               print(f"  ❌ 예측 실패: {img_path.name}: 예측={prediction}, 실제={class_idx}")

        accuracy = 100. * correct / len(image_paths) if len(image_paths) > 0 else 0
        print(f"  클래스 {class_idx}: {correct}/{len(image_paths)} 정답 ({accuracy:.2f}%)")

        total_correct += correct
        total_images += len(image_paths)

    overall_accuracy = 100. * total_correct / total_images if total_images > 0 else 0
    print(f"\n--- {answer_type} / {model_type.upper()} 전체 정확도: {overall_accuracy:.2f}% ({total_correct}/{total_images}) ---")
    return overall_accuracy, total_correct, total_images


def main():
    """메인 실행"""

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"디바이스: {device}")

    base_dir = Path(".")
    # cross-validation 결과가 저장된 폴더로 변경
    model_root = base_dir / "answer_models_cv_results"
    data_root = base_dir # 데이터셋 루트 폴더

    # 평가할 모델 타입과 데이터셋 쌍
    models_to_evaluate = [
        {'answer': 'answer_1', 'type': 'resnet'},
        {'answer': 'answer_2', 'type': 'resnet'},
         {'answer': 'answer_1', 'type': 'efficientnet'}, 
        {'answer': 'answer_2', 'type': 'efficientnet'},
        {'answer': 'answer_1', 'type': 'mobilenet'},
        {'answer': 'answer_2', 'type': 'mobilenet'},
    ]

    results = {}

    for item in models_to_evaluate:
        answer_type = item['answer']
        model_type = item['type']
        
        model_dir = model_root / f"{answer_type}_{model_type}"
        model_path = model_dir / "best_model_overall_cv.pth"


        data_dir = data_root / answer_type

        if not model_path.exists():
            print(f"❌ 모델 파일 없음: {model_path}. {answer_type} / {model_type} 평가를 건너뜁니다.")
            continue

        if not data_dir.exists():
            print(f"❌ 데이터 폴더 없음: {data_dir}")
            continue

        try:
            model = load_answer_model(model_path, model_type=model_type, device=device)

            acc, correct, total = evaluate_dataset(model, data_dir, model_type, answer_type, device)
            results[f"{answer_type}_{model_type}"] = {'acc': acc, 'correct': correct, 'total': total}
        except Exception as e:
            print(f"❌ {answer_type} / {model_type} 평가 중 오류 발생: {e}")

    # 최종 결과 요약
    print(f"\n{'='*60}")
    print("🏆 최종 성능 요약")
    print(f"{'='*60}")

    for answer_type in ['answer_1', 'answer_2']:
        print(f"\n📊 {answer_type} 성능:")
        evaluated_models = [m_type for m_type in ['resnet', 'efficientnet', 'mobilenet'] if f"{answer_type}_{m_type}" in results]
        if evaluated_models:
            for model_type in evaluated_models:
                key = f"{answer_type}_{model_type}"
                print(f"  {model_type.upper()}: {results[key]['acc']:.2f}%")
        else:
            print("  평가된 모델 없음")


if __name__ == "__main__":
    main()
"""

SyntaxError: invalid syntax (3505671357.py, line 15)